# Bước 3: Đọc hiểu và Trả lời (FiD Reader - Table 4 Replication)
Notebook này tái hiện lại chuẩn xác Table 4 của bài báo:
- Sử dụng 2 mô hình Retriever đại diện: **Contriever** (Unsupervised) và **DPR** (Supervised).
- Truy xuất Top-5 và Top-20 Passages cho 3 cấp độ: Passage, Sentence, Proposition.
- Đưa các đoạn văn vào mô hình đọc hiểu FiD (**nq_reader_large**) của Facebook Research.
- Đo lường điểm Exact Match (EM) và xuất kết quả ra file CSV và Markdown.

In [ ]:
!pip install sentence-transformers faiss-cpu pandas pyarrow transformers accelerate huggingface_hub tabulate

## 0. Nạp HuggingFace Token

In [ ]:
from huggingface_hub import login
import os

# ĐIỀN TOKEN CỦA BẠN VÀO ĐÂY
TOKEN_CUA_BAN = 'hf_xxxxxxxxx'
login(token=TOKEN_CUA_BAN)
os.environ["HF_TOKEN"] = TOKEN_CUA_BAN
print("Đã nạp Token thành công!")

## 1. Nạp Dữ liệu từ Bước 1

In [ ]:
import pandas as pd
import numpy as np
import os

# Kiểm tra thư mục Input
input_dir = '/kaggle/input/output-notebook1-seg'  # SỬA ĐƯỜNG DẪN NÀY NẾU KHÁC

if not os.path.exists(f'{input_dir}/test_questions.jsonl'):
    # Thử tìm tự động trong /kaggle/input/
    found = False
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'test_questions.jsonl' in files:
            input_dir = root
            found = True
            print(f"--> Đã tự động tìm thấy thư mục dữ liệu tại: {input_dir}")
            break
    if not found:
        raise FileNotFoundError(f"Không tìm thấy test_questions.jsonl! Vui lòng kiểm tra lại Input Data trong Kaggle.")

questions_df = pd.read_json(f'{input_dir}/test_questions.jsonl', lines=True)
corpus = {
    'passage': pd.read_parquet(f'{input_dir}/corpus_passage.parquet'),
    'sentence': pd.read_parquet(f'{input_dir}/corpus_sentence.parquet'),
    'proposition': pd.read_parquet(f'{input_dir}/corpus_proposition.parquet')
}

def extract_exact_title(row):
    span = row['metadata']['title_span']
    return row['contents'][span[0]:span[1]]

passage_title_dict = dict(zip(corpus['passage']['id'], corpus['passage'].apply(extract_exact_title, axis=1)))
passage_dict = dict(zip(corpus['passage']['id'], corpus['passage']['contents']))
print(f"Đã nạp {len(questions_df)} câu hỏi và {len(passage_dict)} bài báo!")

## 2. Truy xuất Top-20 Passages (Contriever & DPR) với cơ chế Lưu Cache chống mất dữ liệu

In [ ]:
import torch
import faiss
import gc
import json
from sentence_transformers import SentenceTransformer, models

cache_path = '/kaggle/working/retrieved_results_table4.json'

if os.path.exists(cache_path):
    print("--> Tìm thấy file cache kết quả truy xuất! Đang tải lại trực tiếp...")
    with open(cache_path, 'r', encoding='utf-8') as f:
        retrieved_results = json.load(f)
    print("--> Tải cache thành công! Bỏ qua bước mã hóa tốn thời gian.")
else:
    def load_rigorous_model(model_name_or_path):
        if 'sentence-transformers' in model_name_or_path:
            return SentenceTransformer(model_name_or_path, device='cpu')
        word_embedding_model = models.Transformer(model_name_or_path)
        dim = word_embedding_model.get_word_embedding_dimension()
        if 'contriever' in model_name_or_path.lower():
            pooler = models.Pooling(dim, pooling_mode_mean_tokens=True, pooling_mode_cls_token=False)
        else:
            pooler = models.Pooling(dim, pooling_mode_mean_tokens=False, pooling_mode_cls_token=True)
        return SentenceTransformer(modules=[word_embedding_model, pooler], device='cpu')

    models_config = {
        'Contriever': 'facebook/contriever',
        'DPR': ('sentence-transformers/facebook-dpr-question_encoder-multiset-base', 
                'sentence-transformers/facebook-dpr-ctx_encoder-multiset-base')
    }

    retrieved_results = {}

    for model_name, paths in models_config.items():
        print(f"\n{'='*40}\nĐang truy xuất cho: {model_name}\n{'='*40}")
        retrieved_results[model_name] = {}
        
        if model_name == 'DPR':
            q_model = load_rigorous_model(paths[0])
            c_model = load_rigorous_model(paths[1])
        else:
            q_model = load_rigorous_model(paths)
            c_model = q_model

        pool = c_model.start_multi_process_pool()
        
        print("Encoding questions...")
        q_emb = q_model.encode(questions_df['question'].tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
        q_emb = np.ascontiguousarray(q_emb, dtype=np.float32)
        
        for level in ['passage', 'sentence', 'proposition']:
            print(f"--> Đang nhúng & tìm kiếm cho cấp độ: {level} ({len(corpus[level])} dòng)")
            texts = corpus[level]['contents'].tolist()
            b_size = 64 if level == 'passage' else 128
            
            c_emb = c_model.encode(texts, pool=pool, batch_size=b_size)
            c_emb = np.ascontiguousarray(c_emb, dtype=np.float32)
            faiss.normalize_L2(c_emb)
            
            index = faiss.IndexFlatIP(c_emb.shape[1])
            index.add(c_emb)
            search_k = 20 if level == 'passage' else 200
            scores, indices = index.search(q_emb, search_k)
            
            level_results = []
            for i, (row_scores, row_indices) in enumerate(zip(scores, indices)):
                passage_scores = {}
                for score, idx in zip(row_scores, row_indices):
                    unit_id = corpus[level].iloc[idx]['id']
                    p_id = unit_id if level == 'passage' else unit_id.rsplit('-', 1)[0]
                    if p_id not in passage_scores:
                        passage_scores[p_id] = float(score)
                top20_ids = [p_id for p_id, _ in sorted(passage_scores.items(), key=lambda x: x[1], reverse=True)[:20]]
                level_results.append(top20_ids)
                
            retrieved_results[model_name][level] = level_results
            del c_emb, index
            gc.collect()
            
        c_model.stop_multi_process_pool(pool)
        if model_name == 'DPR':
            del c_model, q_model
        else:
            del q_model, c_model
        gc.collect()
        torch.cuda.empty_cache()
        
    print("\nHoàn tất truy xuất Top 20 Passages cho cả 2 Models!")
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(retrieved_results, f)
    print(f"--> Đã lưu file cache thành công: {cache_path}")

## 3. Tải Checkpoint nq_reader_large của FiD từ Facebook Research

In [ ]:
if not os.path.exists('nq_reader_large/pytorch_model.bin'):
    print("Đang tải nq_reader_large.tar.gz từ máy chủ Meta/Facebook...")
    !wget -nc https://dl.fbaipublicfiles.com/FiD/pretrained_models/nq_reader_large.tar.gz
    print("Đang giải nén checkpoint...")
    !tar -xvzf nq_reader_large.tar.gz
else:
    print("Đã có sẵn thư mục checkpoint nq_reader_large!")

## 4. Nạp Trọng số FiD vào Kiến trúc T5-large Chuẩn (Khắc phục triệt để lỗi xung đột Transformers)

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
from transformers.modeling_outputs import BaseModelOutput

print("Đang nạp Tokenizer & Kiến trúc T5-large...")
tokenizer = T5Tokenizer.from_pretrained('t5-large', model_max_length=250)
model = T5ForConditionalGeneration.from_pretrained('t5-large')

print("Đang đọc và chuyển đổi trọng số từ nq_reader_large/pytorch_model.bin...")
raw_state_dict = torch.load('nq_reader_large/pytorch_model.bin', map_location='cpu')

# Chuyển đổi tên các trọng số từ cấu trúc đóng gói riêng của Facebook sang T5 chuẩn
clean_state_dict = {}
for k, v in raw_state_dict.items():
    new_k = k
    if new_k.startswith("encoder.encoder."):
        new_k = new_k.replace("encoder.encoder.", "encoder.", 1)
    new_k = new_k.replace(".module.", ".")
    clean_state_dict[new_k] = v

missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)
print(f"--> Đã nạp thành công trọng số FiD vào T5-large!")
print(f"    Số layer khớp: {len(clean_state_dict) - len(unexpected)} | Layer bổ sung bỏ qua: {len(unexpected)}")

# Giải phóng RAM và đưa model lên GPU
del raw_state_dict, clean_state_dict
gc.collect()

model = model.to('cuda').eval()
print("--> FiD Reader đã sẵn sàng trong GPU VRAM!")

## 5. Chạy Đọc hiểu (FiD Inference) & Đánh giá Exact Match (Table 4)

In [ ]:
import re
import string
from tqdm.auto import tqdm

def normalize_answer(s):
    """Chuẩn hóa theo quy chuẩn chung của SQuAD / NQ"""
    if s is None:
        return ""
    text = str(s).lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = ''.join(ch for ch in text if ch not in set(string.punctuation))
    return ' '.join(text.split())

def exact_match(prediction, ground_truths):
    if not isinstance(ground_truths, list):
        ground_truths = [ground_truths]
    pred_norm = normalize_answer(prediction)
    return max([int(pred_norm == normalize_answer(gt)) for gt in ground_truths], default=0)

def format_fid_input(question, p_ids):
    passages_str = []
    for pid in p_ids:
        title = passage_title_dict.get(pid, "")
        text = passage_dict.get(pid, "")
        passages_str.append(f"question: {question} title: {title} context: {text}")
    return passages_str

def run_fid_inference(model, tokenizer, question, p_ids, max_passages=5):
    selected_pids = p_ids[:max_passages]
    if len(selected_pids) == 0:
        return ""
        
    passages_str = format_fid_input(question, selected_pids)
    inputs = tokenizer(passages_str, return_tensors='pt', padding=True, truncation=True, max_length=250)
    input_ids = inputs['input_ids'].to('cuda')
    attention_mask = inputs['attention_mask'].to('cuda')
    
    N, L = input_ids.shape
    
    with torch.no_grad():
        # Mã hóa độc lập N đoạn văn qua Encoder của T5
        enc_out = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
        
        # Gộp (Fusion) các trạng thái ẩn thành chuỗi dài (1, N*L, Dim) cho Decoder
        fusion_hidden_states = enc_out.last_hidden_state.view(1, N * L, -1)
        fusion_attention_mask = attention_mask.view(1, N * L)
        
        wrapped_enc_out = BaseModelOutput(last_hidden_state=fusion_hidden_states)
        
        # Sinh câu trả lời qua Decoder
        gen_out = model.generate(
            encoder_outputs=wrapped_enc_out,
            attention_mask=fusion_attention_mask,
            max_new_tokens=20
        )
        
    return tokenizer.decode(gen_out[0], skip_special_tokens=True)

all_em_scores = []
n_questions = len(questions_df)

for model_name in ['Contriever', 'DPR']:
    for level in ['passage', 'sentence', 'proposition']:
        print(f"\n{'='*40}\nĐang chạy FiD Reader: {model_name} - {level.upper()}\n{'='*40}")
        em_5, em_20 = 0, 0
        
        for i, row in tqdm(questions_df.iterrows(), total=n_questions):
            question = row['question']
            gold_answers = row['answers']
            p_ids = retrieved_results[model_name][level][i]
            
            # Đọc hiểu với Top-5
            ans_5 = run_fid_inference(model, tokenizer, question, p_ids, max_passages=5)
            em_5 += exact_match(ans_5, gold_answers)
            
            # Đọc hiểu với Top-20
            ans_20 = run_fid_inference(model, tokenizer, question, p_ids, max_passages=20)
            em_20 += exact_match(ans_20, gold_answers)
            
            if (i + 1) % 100 == 0:
                torch.cuda.empty_cache()
                
        metrics = {
            'Retriever': model_name,
            'Granularity': level,
            'EM_Top5': round(em_5 / n_questions * 100, 2),
            'EM_Top20': round(em_20 / n_questions * 100, 2)
        }
        all_em_scores.append(metrics)
        print(f"Kết quả: {model_name} [{level}] -> EM@5 = {metrics['EM_Top5']}% | EM@20 = {metrics['EM_Top20']}%")
        
        # Lưu tạm kết quả sau mỗi mốc
        pd.DataFrame(all_em_scores).to_csv('/kaggle/working/table4_fid_em.csv', index=False)

print("\nHOÀN TẤT ĐÁNH GIÁ FID TOÀN BỘ!")

## 6. Tổng hợp & Xuất Báo cáo Markdown (Table 4 Replication)

In [ ]:
results_df = pd.DataFrame(all_em_scores)
results_df.to_csv('/kaggle/working/table4_fid_em.csv', index=False)

md_content = "# Báo cáo Table 4: Exact Match của Fusion-in-Decoder (FiD)\n\n"
md_content += "Bảng dưới đây đo lường năng lực đọc hiểu thực tế của mô hình FiD (`nq_reader_large`) "
md_content += "dựa trên các đoạn văn bản truy xuất từ Contriever và DPR ở 3 cấp độ (Granularity):\n\n"
md_content += results_df.to_markdown(index=False)

with open('/kaggle/working/table4_fid_em.md', 'w', encoding='utf-8') as f:
    f.write(md_content)

display(results_df)
print("\nĐã tạo thành công table4_fid_em.csv và table4_fid_em.md!")